## Практическая работа. Подготовка данных

Модель учится только на том, что ей дали. Если в таблице пропуски, опечатки и дубликаты – модель выучит мусор
(**garbage in – garbage out**). Поэтому перед обучением данные чистят и приводят к виду, удобному для модели.

**Конвейер подготовки** – шаги идут именно в таком порядке:

| Шаг | Что делаем | Формула / правило |
|---|---|---|
| 1. Осмотр | считаем пропуски и уникальные значения | пропуск – это `None` |
| 2. Дубликаты и разнобой | удаляем повторы строк, приводим записи к одному виду | `' Физмат '` → `'физмат'` |
| 3. Средние | среднее, медиана, мода | $\bar x = \dfrac{1}{n}\sum_{i=1}^{n} x_i$; медиана – середина отсортированного списка; мода – самое частое значение |
| 4. Выбросы | ищем значения далеко от остальных | $IQR = Q_3 - Q_1$, норма: $[\,Q_1 - 1{,}5\cdot IQR;\ Q_3 + 1{,}5\cdot IQR\,]$ |
| 5. Пропуски | заполняем медианой (числа) или модой (категории) | медиана устойчива к выбросам |
| 6. Кодирование | текст → числа | one-hot: «физмат» → `[0, 1, 0]` |
| 7. Разделение | обучающая и тестовая выборки | обычно 70 / 30 |
| 8. Масштабирование | признаки к одному масштабу | min-max: $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$; стандартизация: $z = \dfrac{x - \mu}{\sigma}$ |

**Квантиль.** $Q_q$ – число, левее которого лежит доля $q$ значений. Отсортируем $n$ значений $v_0 \le v_1 \le \dots \le v_{n-1}$,
найдем позицию $pos = (n-1)\cdot q$, ее целую часть $i$ и дробную часть $f$: $\;Q_q = v_i + f\cdot(v_{i+1} - v_i)$.
$Q_1 = Q_{0{,}25}$, медиана $= Q_{0{,}5}$, $Q_3 = Q_{0{,}75}$.

**Стандартное отклонение** – средний «разброс» вокруг среднего: $\sigma = \sqrt{\dfrac{1}{n}\sum_{i=1}^{n}(x_i - \mu)^2}$.

> **Главное правило.** Все, что вычисляется по данным (медиана, границы выбросов, $x_{min}$, $x_{max}$, $\mu$, $\sigma$),
> считается **только по обучающей выборке**, а потом применяется и к обучающей, и к тестовой. Иначе – утечка данных.

> **Правило работы.** Внутри функций – только списки, словари, циклы, `math` и `random`. Без numpy, pandas и sklearn:
> они нужны только в готовых ячейках для загрузки данных и сверки.

In [ ]:
# Ячейка готова. Библиотеки нужны только для загрузки данных, графиков и сверки ответов.
import math
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

# Сырая анкета 11 учеников: часы подготовки, сон перед зачетом, балл за пробник, профиль класса, сдал ли зачет.
# None – пропуск. В таблице спрятаны все типичные проблемы – найдите их.
raw = [
    {'часы': 1,    'сон': 8,    'пробник': 35,   'профиль': 'физмат',       'сдал': 0},
    {'часы': 2,    'сон': 6,    'пробник': 40,   'профиль': 'Физмат',       'сдал': 0},
    {'часы': None, 'сон': 7,    'пробник': 52,   'профиль': 'химбио',       'сдал': 0},
    {'часы': 3,    'сон': 9,    'пробник': 61,   'профиль': ' гум',         'сдал': 1},
    {'часы': 4,    'сон': 50,   'пробник': 48,   'профиль': 'физмат ',      'сдал': 0},
    {'часы': 5,    'сон': 6,    'пробник': None, 'профиль': 'Гуманитарный', 'сдал': 1},
    {'часы': 6,    'сон': 8,    'пробник': 73,   'профиль': 'ф/м',          'сдал': 1},
    {'часы': 7,    'сон': None, 'пробник': 70,   'профиль': 'гуманитарный', 'сдал': 1},
    {'часы': 8,    'сон': 6,    'пробник': 85,   'профиль': 'ХимБио',       'сдал': 1},
    {'часы': 9,    'сон': 8,    'пробник': 90,   'профиль': 'физмат',       'сдал': 1},
    {'часы': 2,    'сон': 6,    'пробник': 40,   'профиль': 'Физмат',       'сдал': 0},
]
columns = ['часы', 'сон', 'пробник', 'профиль', 'сдал']

def column(rows, name):
    """Столбец таблицы как список значений."""
    return [row[name] for row in rows]

def set_column(rows, name, values):
    """Новая таблица, в которой столбец name заменен на values (исходная не меняется)."""
    return [{**row, name: v} for row, v in zip(rows, values)]

def show(rows, cols=None):
    """Печать таблицы: пропуск показан как «–»."""
    cols = cols or list(rows[0].keys())
    width = {c: max(len(str(c)), *(len(repr(r[c])) for r in rows)) for c in cols}
    print('  #  ' + ' | '.join(str(c).ljust(width[c]) for c in cols))
    for i, r in enumerate(rows, 1):
        print(f'{i:>3}  ' + ' | '.join(('–' if r[c] is None else repr(r[c])).ljust(width[c]) for c in cols))

show(raw)

**Задание П1. Осмотр данных.** Прежде чем чистить, нужно понять, что сломано. Напишите две функции:

1. `count_missing(values)` – сколько в списке значений `None`;
2. `unique_values(values)` – список различных значений в порядке первого появления (пропуски не включать).

Критерий правильности – ожидаемые значения в выводе ячейки.

In [ ]:
# ЗАДАНИЕ П1. Заполните пропуски.
def count_missing(values):
    count = 0
    ...            # <-- увеличьте счетчик для каждого None
    return count

def unique_values(values):
    result = []
    ...            # <-- добавляйте значение, если его еще нет в result и это не None
    return result

for c in columns:
    print(f'{c:<8} пропусков: {count_missing(column(raw, c))}')
print('ожидается: часы 1, сон 1, пробник 1, профиль 0, сдал 0')
print()
print('Варианты записи профиля:', unique_values(column(raw, 'профиль')))
print('Сколько вариантов:', len(unique_values(column(raw, 'профиль'))), '  ожидается 9')

**Вопрос П1.** Посмотрите на таблицу и выпишите все проблемы, которые видите: пропуски, подозрительные
значения, повторы, разнобой. Какое значение сна выглядит невозможным и какая опечатка могла его породить?
Сколько на самом деле профилей в классе, если вариантов записи 9?

*Ваш ответ:*

**Задание П2. Дубликаты и разнобой в записи.**

1. `remove_duplicates(rows)` – новая таблица без повторяющихся строк; из одинаковых строк оставляем **первую**
   (две строки-словаря равны, если `row1 == row2`);
2. `clean_category(text, synonyms)` – приводит запись к единому виду: убрать пробелы по краям (`strip`),
   перевести в нижний регистр (`lower`), затем, если результат есть в словаре `synonyms`, заменить его на полное название.

In [ ]:
# ЗАДАНИЕ П2. Заполните пропуски.
def remove_duplicates(rows):
    result = []
    ...            # <-- добавляйте строку, только если ее еще нет в result
    return result

def clean_category(text, synonyms):
    text = ...     # <-- strip и lower
    ...            # <-- замена по словарю synonyms
    return text

synonyms = {'ф/м': 'физмат', 'гум': 'гуманитарный'}

data = remove_duplicates(raw)
print('Строк было:', len(raw), ' стало:', len(data), '  ожидается 11 -> 10')

data = set_column(data, 'профиль', [clean_category(p, synonyms) for p in column(data, 'профиль')])
print('Профили после чистки:', unique_values(column(data, 'профиль')))
print("ожидается: ['физмат', 'химбио', 'гуманитарный']")

**Вопрос П2.** Чем опасен дубликат для модели? Представьте, что одна и та же строка повторилась 100 раз.
Что будет, если не привести профили к единому виду: сколько «разных профилей» увидит модель?

*Ваш ответ:*

**Задание П3. Среднее, медиана, мода.** Все три функции **пропускают `None`**.

1. `mean(values)` – среднее: $\bar x = \dfrac{1}{n}\sum x_i$;
2. `median(values)` – медиана: отсортировать; при нечетном $n$ – средний элемент, при четном – полусумма двух средних;
3. `mode(values)` – самое частое значение (подсчитайте частоты в словаре, подсказка: `max(counts, key=counts.get)`).

In [ ]:
# ЗАДАНИЕ П3. Заполните пропуски.
def known(values):
    """Готовая функция: значения без пропусков."""
    return [v for v in values if v is not None]

def mean(values):
    v = known(values)
    return ...     # <-- сумма / количество

def median(values):
    v = sorted(known(values))
    n = len(v)
    m = n // 2
    ...            # <-- два случая: n нечетное и n четное

def mode(values):
    counts = {}
    ...            # <-- подсчет частот
    return ...     # <-- самое частое значение

sleep = column(data, 'сон')
print('Сон:', sleep)
print(f'mean(сон)   = {mean(sleep):.2f}    ожидается 12.00')
print(f'median(сон) = {median(sleep)}       ожидается 8')
print(f'median([4, 1, 3, 2]) = {median([4, 1, 3, 2])}   ожидается 2.5')
print(f"mode(профиль) = {mode(column(data, 'профиль'))}   ожидается физмат")

**Вопрос П3.** По среднему выходит, что ученики спят 12 часов. Почему среднее так ошиблось, а медиана – нет?
Для какого признака из таблицы подходит только мода и почему среднее для него посчитать нельзя?

*Ваш ответ:*

**Задание П4. Выбросы.**

1. `quantile(values, q)` – квантиль по формуле из начала ноутбука: отсортировать известные значения,
   $pos = (n-1)\cdot q$, $i$ = `int(pos)`, $f = pos - i$, $\;Q_q = v_i + f\cdot(v_{i+1}-v_i)$
   (если $i$ – последний индекс, вернуть $v_i$);
2. `iqr_bounds(values, k=1.5)` – границы нормы $(\,Q_1 - k\cdot IQR,\ Q_3 + k\cdot IQR\,)$;
3. `outliers_to_none(values, low, high)` – новый список, где значения вне $[low;\ high]$ заменены на `None`
   (пропуски остаются пропусками).

Выброс мы не удаляем вместе со строкой, а превращаем в пропуск – остальные признаки ученика еще пригодятся.

In [ ]:
# ЗАДАНИЕ П4. Заполните пропуски.
def quantile(values, q):
    v = sorted(known(values))
    pos = ...      # <-- позиция
    i = int(pos)
    f = pos - i
    if i + 1 >= len(v):
        return v[i]
    return ...     # <-- интерполяция между v[i] и v[i + 1]

def iqr_bounds(values, k=1.5):
    q1, q3 = ..., ...   # <-- два квартиля
    iqr = q3 - q1
    return ..., ...     # <-- нижняя и верхняя границы

def outliers_to_none(values, low, high):
    result = []
    for v in values:
        ...        # <-- None остается None; вне границ – None; иначе само значение
    return result

print(f"Q1 = {quantile(sleep, 0.25)}, медиана = {quantile(sleep, 0.5)}, Q3 = {quantile(sleep, 0.75)}   ожидается 6.0, 8.0, 8.0")
print('quantile([1, 2, 3, 4], 0.25) =', quantile([1, 2, 3, 4], 0.25), '  ожидается 1.75')
low, high = iqr_bounds(sleep)
print(f'Границы нормы для сна: [{low}; {high}]   ожидается [3.0; 11.0]')
sleep_clean = outliers_to_none(sleep, low, high)
print('Сон после:', sleep_clean)
print('Границы для пробника:', iqr_bounds(column(data, 'пробник')), '  ожидается (10.5, 110.5) – выбросов нет')

In [ ]:
# Ящик с усами до и после (ячейка готова)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].boxplot(known(sleep), vert=False)
ax[0].set_title('Сон: до очистки')
ax[1].boxplot(known(sleep_clean), vert=False)
ax[1].set_title('Сон: выброс убран')
for a in ax:
    a.set_xlabel('часы сна'); a.set_yticks([])
plt.tight_layout(); plt.show()

# Сверка квантилей с numpy (метод по умолчанию – тот же, что в нашей формуле)
import numpy as np
print('numpy:', np.quantile(known(sleep), [0.25, 0.5, 0.75]))

**Вопрос П4.** Почему границы строятся от квартилей, а не от минимума и максимума? Всегда ли выброс – это
ошибка? Приведите пример настоящего, не ошибочного выброса (например, рост баскетболиста в классе).

*Ваш ответ:*

**Задание П5. Заполнение пропусков.** Напишите `fill_missing(values, fill_value)` – новый список, где каждый
`None` заменен на `fill_value`. Затем соберите чистую таблицу:

- в столбце «сон» сначала выброс → `None` (это уже сделано в `sleep_clean`), затем пропуски → медиана;
- в столбцах «часы» и «пробник» пропуски → медиана;
- в столбце «профиль» пропусков нет, но для категорий правило такое: пропуск → мода.

In [ ]:
# ЗАДАНИЕ П5. Заполните пропуски.
def fill_missing(values, fill_value):
    return ...     # <-- None заменить на fill_value, остальное оставить

clean = set_column(data, 'сон', fill_missing(sleep_clean, median(sleep_clean)))
for c in ['часы', 'пробник']:
    col = column(clean, c)
    clean = ...    # <-- заполните пропуски медианой этого столбца

show(clean)
print()
print('Пропусков осталось:', sum(count_missing(column(clean, c)) for c in columns), '  ожидается 0')
print('Сон:', column(clean, 'сон'))
print('ожидается [8, 6, 7, 9, 7.5, 6, 8, 7.5, 6, 8]')

**Вопрос П5.** Ученик №3 на самом деле готовился 2 часа, а мы подставили медиану 5. Значит ли это, что
заполнение – плохая идея? Когда лучше удалить строку целиком, а когда – весь столбец?

*Ваш ответ:*

**Задание П6. Кодирование категорий.** Модель умеет работать только с числами.

1. `label_encode(values)` – каждой категории номер по алфавиту: возвращает `(коды, словарь)`,
   например `['б', 'а', 'б']` → `([1, 0, 1], {'а': 0, 'б': 1})`;
2. `one_hot(values)` – каждая категория превращается в отдельный столбец из 0 и 1: возвращает
   `(список строк-векторов, список категорий по алфавиту)`, например `['б', 'а']` → `([[0, 1], [1, 0]], ['а', 'б'])`.

In [ ]:
# ЗАДАНИЕ П6. Заполните пропуски.
def label_encode(values):
    categories = sorted(unique_values(values))
    mapping = ...  # <-- словарь {категория: номер}
    codes = ...    # <-- номер для каждого значения
    return codes, mapping

def one_hot(values):
    categories = sorted(unique_values(values))
    vectors = []
    for v in values:
        ...        # <-- вектор длины len(categories): 1 на месте категории v, остальные 0
    return vectors, categories

profiles = column(clean, 'профиль')
codes, mapping = label_encode(profiles)
print('label_encode:', mapping)
print("ожидается    {'гуманитарный': 0, 'физмат': 1, 'химбио': 2}")
vectors, cats = one_hot(profiles)
print('one_hot, категории:', cats)
for p, vec in list(zip(profiles, vectors))[:3]:
    print(f'  {p:<13} -> {vec}')
print('ожидается: физмат -> [0, 1, 0], химбио -> [0, 0, 1]')

In [ ]:
# Итог: готовая числовая таблица X и ответы y (ячейка готова)
X_toy = [[r['часы'], r['сон'], r['пробник']] + vec for r, vec in zip(clean, vectors)]
y_toy = column(clean, 'сдал')
toy_features = ['часы', 'сон', 'пробник'] + ['проф_' + c for c in cats]
print(toy_features)
for x, t in zip(X_toy, y_toy):
    print(x, '->', t)

**Вопрос П6.** После `label_encode` «химбио» = 2, «гуманитарный» = 0. Какую ложную закономерность может
«увидеть» kNN или логистическая регрессия? Почему one-hot этой проблемы не создает, но при 500 категориях
(например, «город») становится неудобным?

*Ваш ответ:*

**Задание П7. Разделение и масштабирование.**

1. `train_test_split(X, y, test_size=0.3, seed=42)` – перемешать номера строк `random.Random(seed).shuffle(idx)`,
   первые `round(n * test_size)` номеров – в тест, остальные – в обучение; вернуть `X_train, X_test, y_train, y_test`;
2. `fit_min_max(values)` → `(x_min, x_max)` и `apply_min_max(values, params)` → $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$;
3. `fit_standard(values)` → `(mu, sigma)` и `apply_standard(values, params)` → $z = \dfrac{x - \mu}{\sigma}$,
   где $\sigma = \sqrt{\dfrac{1}{n}\sum (x_i - \mu)^2}$.

`fit_...` **изучает** параметры (только на обучающей!), `apply_...` **применяет** их к любым данным.

In [ ]:
# ЗАДАНИЕ П7. Заполните пропуски.
def train_test_split(X, y, test_size=0.3, seed=42):
    idx = list(range(len(X)))
    random.Random(seed).shuffle(idx)
    n_test = round(len(X) * test_size)
    test_idx, train_idx = ..., ...   # <-- срезы списка idx
    X_train = [X[i] for i in train_idx]
    X_test = ...
    y_train = ...
    y_test = ...
    return X_train, X_test, y_train, y_test

def fit_min_max(values):
    return ..., ...

def apply_min_max(values, params):
    x_min, x_max = params
    return ...

def fit_standard(values):
    mu = ...
    sigma = ...    # <-- math.sqrt от среднего квадрата отклонений
    return mu, sigma

def apply_standard(values, params):
    mu, sigma = params
    return ...

p = fit_min_max([2, 4, 6, 10])
print('min-max:', p, apply_min_max([2, 4, 6, 10], p), '  ожидается (2, 10) [0.0, 0.25, 0.5, 1.0]')
print('на новых данных:', apply_min_max([0, 12], p), '  ожидается [-0.25, 1.25] – выход за [0; 1] – это нормально')
p = fit_standard([2, 4, 4, 4, 5, 5, 7, 9])
print('standard:', p, '  ожидается (5.0, 2.0)')
print(apply_standard([3, 5, 9], p), '  ожидается [-1.0, 0.0, 2.0]')
Xa, Xb, ya, yb = train_test_split(X_toy, y_toy, test_size=0.3)
print('Разделение 10 учеников:', len(Xa), 'в обучении,', len(Xb), 'в тесте   ожидается 7 и 3')

**Вопрос П7.** Почему нельзя сначала стандартизировать всю таблицу, а потом разделить ее на обучение и тест?
Сравните с контрольной: что будет, если учитель покажет задания заранее? Почему `apply_min_max` на тестовых
данных может дать число больше 1?

*Ваш ответ:*

**Задание П8. Зачем все это: эксперимент на Wine.** Датасет Wine – 178 вин трех сортов, 13 химических признаков.
Признаки живут в очень разных масштабах: оттенок (`hue`) – от 0,5 до 1,7, а пролин (`proline`) – от 278 до 1680.
kNN измеряет расстояние, и пролин «перекрикивает» все остальные признаки.

Ниже готовы: загрузка, kNN на чистом Python и функция `scale_columns`, которая применяет ваши `fit_...`/`apply_...`
к каждому столбцу. Заполните три строки и сравните точность.

In [ ]:
# Ячейка готова: данные и kNN
wine = load_wine()
X_wine, y_wine = wine.data.tolist(), wine.target.tolist()
wine_features = list(wine.feature_names)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_wine, y_wine, test_size=0.3)

def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for x in X_test:
        dist = sorted((sum((a - b) ** 2 for a, b in zip(x, xt)), label) for xt, label in zip(X_train, y_train))
        votes = {}
        for _, label in dist[:k]:
            votes[label] = votes.get(label, 0) + 1
        preds.append(max(votes, key=votes.get))
    return preds

def accuracy(y_true, y_pred):
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)

def scale_columns(X_train, X_test, fit, apply):
    """Параметры каждого столбца – по обучающей, применение – к обеим выборкам."""
    A, B = [list(r) for r in X_train], [list(r) for r in X_test]
    for j in range(len(A[0])):
        params = fit([r[j] for r in A])
        for rows in (A, B):
            new = apply([r[j] for r in rows], params)
            for r, v in zip(rows, new):
                r[j] = v
    return A, B

print('Обучающая:', len(Xw_train), ' тестовая:', len(Xw_test))
for name in ['hue', 'proline']:
    j = wine_features.index(name)
    col = [r[j] for r in Xw_train]
    print(f'{name:<8} от {min(col)} до {max(col)}')

In [ ]:
# ЗАДАНИЕ П8. Заполните три строки.
acc_raw = ...                       # <-- kNN на исходных признаках
A, B = scale_columns(Xw_train, Xw_test, fit_min_max, apply_min_max)
acc_minmax = ...                    # <-- kNN после min-max
A, B = ...                          # <-- стандартизация
acc_std = accuracy(yw_test, knn_predict(A, yw_train, B))

print(f'Без масштабирования: {acc_raw:.3f}   ожидается 0.623')
print(f'Min-max:             {acc_minmax:.3f}   ожидается 0.925')
print(f'Стандартизация:      {acc_std:.3f}   ожидается 0.943')

plt.figure(figsize=(6, 3))
plt.bar(['без масштаба', 'min-max', 'стандартизация'], [acc_raw, acc_minmax, acc_std], color=['#9a9aa3', '#b8ab84', '#1c1c22'])
plt.ylim(0, 1); plt.ylabel('точность kNN'); plt.title('Wine: одна модель, разная подготовка')
plt.show()

In [ ]:
# Сверка со sklearn (ячейка готова): ваша стандартизация должна совпасть со StandardScaler
from sklearn.preprocessing import StandardScaler
sk = StandardScaler().fit(Xw_train)
A, _ = scale_columns(Xw_train, Xw_test, fit_standard, apply_standard)
diff = max(abs(a - b) for ra, rb in zip(A, sk.transform(Xw_train).tolist()) for a, b in zip(ra, rb))
print(f'Максимальное расхождение с StandardScaler: {diff:.2e}   (должно быть около 1e-15)')

**Вопрос П8.** Модель одна и та же, данные те же – почему точность выросла с 62 % до 94 %?
Нужно ли масштабирование дереву решений? Объясните через то, как дерево задает вопросы.

*Ваш ответ:*

**Задание П9. Весь конвейер на «грязном» Wine.** Готовая функция `make_dirty` портит данные так, как это бывает
в жизни: примерно 5 % значений пропадают, а 2 % умножаются на 10 (лишний ноль при вводе).
kNN с пропусками вообще не запустится – расстояние до `None` не посчитать.

Напишите `prepare(X_train, X_test, remove_outliers=True)`. Для **каждого столбца** $j$:

1. взять столбец обучающей и тестовой выборки;
2. если `remove_outliers` – посчитать `iqr_bounds` **по обучающей** и превратить выбросы в `None` в обеих;
3. посчитать медиану **по обучающей** и заполнить пропуски в обеих;
4. стандартизировать: параметры **по обучающей**, применить к обеим.

In [ ]:
# Ячейка готова: портим данные
def make_dirty(X, seed, p_missing=0.05, p_outlier=0.02):
    rnd = random.Random(seed)
    result = []
    for row in X:
        new_row = []
        for v in row:
            u = rnd.random()
            if u < p_missing:
                new_row.append(None)
            elif u < p_missing + p_outlier:
                new_row.append(v * 10)
            else:
                new_row.append(v)
        result.append(new_row)
    return result

D_train, D_test = make_dirty(Xw_train, seed=1), make_dirty(Xw_test, seed=2)
n_cells = len(D_train) * len(D_train[0])
print('Пропусков в обучающей:', sum(count_missing(r) for r in D_train), 'из', n_cells, 'ячеек')

In [ ]:
# ЗАДАНИЕ П9. Заполните пропуски.
def prepare(X_train, X_test, remove_outliers=True):
    A, B = [list(r) for r in X_train], [list(r) for r in X_test]
    for j in range(len(A[0])):
        col_a = [r[j] for r in A]
        col_b = [r[j] for r in B]
        if remove_outliers:
            low, high = ...                         # <-- границы по обучающей
            col_a, col_b = ..., ...                 # <-- выбросы -> None в обеих
        fill = ...                                  # <-- медиана по обучающей
        col_a, col_b = ..., ...                     # <-- заполнить пропуски в обеих
        params = ...                                # <-- параметры стандартизации по обучающей
        col_a, col_b = ..., ...                     # <-- применить к обеим
        for r, v in zip(A, col_a):
            r[j] = v
        for r, v in zip(B, col_b):
            r[j] = v
    return A, B

A, B = prepare(D_train, D_test, remove_outliers=False)
acc_no_out = accuracy(yw_test, knn_predict(A, yw_train, B))
A, B = prepare(D_train, D_test, remove_outliers=True)
acc_full = accuracy(yw_test, knn_predict(A, yw_train, B))

print(f'Чистые данные + стандартизация:          {acc_std:.3f}')
print(f'Грязные: пропуски заполнены, выбросы нет: {acc_no_out:.3f}   ожидается 0.755')
print(f'Грязные: весь конвейер:                   {acc_full:.3f}   ожидается 0.925')

**Вопрос П9.** Почему выбросы так сильно испортили результат, хотя их всего 2 %? Подсказка: вспомните, как
выброс влияет на $\mu$ и $\sigma$, а значит – на все остальные значения столбца после стандартизации.
Что в вашем конвейере «учится», а что только применяется?

*Ваш ответ:*

---
### Задания повышенной сложности

1. **Робастное масштабирование.** Напишите `fit_robust`/`apply_robust`: $x' = \dfrac{x - \text{медиана}}{IQR}$.
   Замените им стандартизацию в `prepare(..., remove_outliers=False)`. Помогает ли оно без удаления выбросов? Почему?
2. **Флаг пропуска.** Иногда сам факт пропуска – информация («не указал доход»). Добавьте к каждой строке
   признак «было ли пропущено значение в столбце $j$» и проверьте, меняется ли точность.
3. **Заполнение соседями.** Заполните пропуск не медианой, а средним значением этого признака у 5 ближайших
   соседей (расстояние – по известным признакам). Сравните с медианой.
4. **Стратифицированное разделение.** Напишите `train_test_split` с сохранением долей классов: делите номера
   каждого класса отдельно. Сравните доли сортов в тесте до и после.
5. **Утечка своими руками.** Посчитайте параметры стандартизации по **всей** таблице Wine, а не по обучающей.
   Изменилась ли точность? Почему на маленьких данных утечка особенно опасна?